# Notebook 03 — Pruebas de hipótesis A/B en AWS SageMaker

**Proyecto:** comparar conversiones de `old_page` (A/control) y `new_page` (B/treatment).  
**Entrada:** `s3://TU_BUCKET/processed/ab_data_clean.csv` del Notebook 01.  
**Salida:** tabla de resultados, figura, conclusiones y archivos en S3.  

> **Alcance:** análisis de las muestras limpiadas del Notebook 01. La eliminación de registros con asignación/página inconsistente puede sesgar resultados causales: estos hallazgos deben interpretarse como un análisis *per-protocol*, no como estimación causal definitiva por intención de tratar. No se puede recuperar un análisis ITT riguroso sin reconstruir la asignación aleatoria original y auditar exposición/eventos.

**Preguntas:** ¿Existe evidencia de diferencia? ¿En qué dirección? ¿De qué magnitud? ¿Es relevante para el negocio?

## 1. Preparar librerías y configurar rutas

Abre el archivo en SageMaker Studio JupyterLab. Cambia `BUCKET`, o usa `USE_S3=False` y coloca el CSV en `data/ab_data_clean.csv`. No se requiere endpoint ni instancia GPU.

In [ ]:
# Descomenta si las librerías no están instaladas:
# %pip install numpy pandas scipy statsmodels matplotlib boto3
from pathlib import Path
from io import BytesIO
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm, chi2_contingency
from statsmodels.stats.proportion import (
    proportions_ztest, confint_proportions_2indep, proportion_confint,
    proportion_effectsize
)
from statsmodels.stats.power import NormalIndPower
from IPython.display import display

BUCKET = "TU_BUCKET"  # <-- Configura tu bucket
USE_S3 = True
INPUT_KEY = "processed/ab_data_clean.csv"
LOCAL_PATH = Path("data/ab_data_clean.csv")
RESULTS_KEY_PREFIX = "results/notebook03/"
OUTPUT_DIR = Path("outputs_notebook_03")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ALPHA = 0.05
EXPECTED_TREATMENT_SHARE = 0.5  # Solo si el experimento estaba planeado 50/50
MDE_ABSOLUTE = 0.01  # ejemplo de diferencia mínima relevante: 1 punto porcentual
RNG = np.random.default_rng(42)
print("Configuración lista")

## 2. Cargar datos limpios y comprobar esquema

Verificamos independencia a nivel de identificador (un registro por usuario), conversiones binarias y coincidencia entre `group` y `landing_page`. Estas comprobaciones **no demuestran aleatorización**, pero detectan errores de estructura.

In [ ]:
if USE_S3:
    if BUCKET == "TU_BUCKET":
        raise ValueError("Configura BUCKET con el nombre de tu bucket antes de continuar")
    import boto3
    s3 = boto3.client("s3")
    obj = s3.get_object(Bucket=BUCKET, Key=INPUT_KEY)
    df = pd.read_csv(BytesIO(obj["Body"].read()))
else:
    df = pd.read_csv(LOCAL_PATH)

required = {"user_id", "group", "landing_page", "converted"}
missing = required.difference(df.columns)
assert not missing, f"Faltan columnas: {missing}"
assert df["user_id"].is_unique, "Usuarios repetidos: revisa Notebook 01"
assert not df[list(required)].isna().any().any(), "Hay nulos en columnas críticas"
assert df["group"].isin(["control", "treatment"]).all()
assert df["converted"].isin([0, 1]).all()
assert ((df.group.eq("control") & df.landing_page.eq("old_page")) |
        (df.group.eq("treatment") & df.landing_page.eq("new_page"))).all()
assert df.group.nunique() == 2, "Se requieren ambos grupos"
print(f"Filas válidas: {len(df):,}")
display(df.head())

## 3. Formular hipótesis y fijar decisión ANTES del contraste

Sea $p_A$ la probabilidad de conversión del control y $p_B$ la del tratamiento:

- $H_0: p_B-p_A=0$ (misma tasa)
- $H_1: p_B-p_A\neq 0$ (diferente tasa)
- Nivel de significancia $\alpha=0.05$, contraste **bilateral**.

**Interpretación del p-value:** probabilidad, bajo $H_0$ y el modelo estadístico, de obtener un resultado al menos tan extremo como el observado. No es la probabilidad de que $H_0$ sea verdadera. **No significativo** no equivale a demostrar igualdad.

El umbral de relevancia de negocio `MDE_ABSOLUTE=0.01` es **ilustrativo**: si el experimento real tiene una MDE predefinida, sustitúyela antes de analizar.

## 4. Tabla de conversiones y balance de asignación

La diferencia de grupos no implica por sí misma un sesgo: primero debemos conocer la proporción de asignación planificada. El análisis de SRM es **diagnóstico** y no reemplaza la inspección del registro de asignación original.

In [ ]:
summary = (df.groupby("group", observed=True)["converted"]
           .agg(usuarios="size", conversiones="sum", tasa="mean")
           .reindex(["control", "treatment"]))
summary["no_conversiones"] = summary["usuarios"] - summary["conversiones"]
summary["tasa_pct"] = 100 * summary["tasa"]
assert summary["usuarios"].gt(0).all()
display(summary)

n_a, n_b = summary["usuarios"].astype(int).to_numpy()
x_a, x_b = summary["conversiones"].astype(int).to_numpy()
p_a, p_b = summary["tasa"].astype(float).to_numpy()
observed_share = n_b/(n_a+n_b)
print(f"Control {n_a:,} / Tratamiento {n_b:,}; proporción tratamiento {observed_share:.4%}")

In [ ]:
# SRM (Sample Ratio Mismatch): compara el reparto real con uno planeado 50/50.
# Si la asignación no era 50/50, modifica EXPECTED_TREATMENT_SHARE.
expected = np.array([1-EXPECTED_TREATMENT_SHARE, EXPECTED_TREATMENT_SHARE]) * (n_a+n_b)
observed = np.array([n_a, n_b])
from scipy.stats import chisquare
srm_stat, srm_p = chisquare(f_obs=observed, f_exp=expected)
print(f"SRM (chi²) = {srm_stat:.4f}; p-value = {srm_p:.6g}")
if srm_p < 0.001:
    print("ALERTA: fuerte evidencia de reparto inesperado; revisar instrumentación/filtrado ANTES de confiar en el test.")
else:
    print("No se detectó SRM fuerte con umbral diagnóstico 0.001; esto no prueba aleatorización correcta.")

**Ejercicio 1:** ¿Cuántos usuarios y cuántas conversiones hay en cada grupo? ¿Qué pasa si la asignación planeada no era 50/50? ¿Podría la limpieza generar SRM aunque la aleatorización original fuera correcta?

## 5. Comparar magnitud observada

Diferencia absoluta: $\hat\Delta=\hat p_B-\hat p_A$ en **puntos porcentuales** si multiplicamos por 100.  
Lift relativo: $100(\hat p_B/\hat p_A-1)$ en %. Evita confundir ambas escalas.

In [ ]:
difference = p_b - p_a
lift = (p_b/p_a - 1) if p_a > 0 else np.nan
print(f"Control:     {p_a:.4%}")
print(f"Tratamiento: {p_b:.4%}")
print(f"Diferencia:  {100*difference:+.4f} puntos porcentuales")
print(f"Lift:        {100*lift:+.4f}%" if np.isfinite(lift) else "Lift indefinido porque control=0")

## 6. Verificación de condiciones para aproximación normal

Para la prueba Z, necesitamos observaciones aproximadamente independientes y suficientes éxitos/fracaso. Comprobaremos recuentos; la independencia y asignación se validan con información del diseño experimental.

In [ ]:
checks = pd.DataFrame({
    "grupo": ["control", "treatment"],
    "conversiones": [x_a, x_b],
    "no_conversiones": [n_a-x_a, n_b-x_b],
})
checks["al_menos_10_cada_categoria"] = (checks.conversiones >= 10) & (checks.no_conversiones >= 10)
display(checks)
assert checks["al_menos_10_cada_categoria"].all(), "Revisar alternativas exactas si las frecuencias son pequeñas"
print("Los recuentos satisfacen la regla práctica >=10 por grupo/categoría.")

## 7. Prueba Z de dos proporciones (bilateral)

Bajo $H_0$, se estima la proporción agrupada $\hat p=(x_A+x_B)/(n_A+n_B)$. El estadístico es:

$$Z=\frac{\hat p_B-\hat p_A}{\sqrt{\hat p(1-\hat p)(1/n_A+1/n_B)}}.$$

`proportions_ztest` devuelve el estadístico y el p-value de dos colas.

In [ ]:
z_stat, p_value = proportions_ztest(
    count=np.array([x_b, x_a]),
    nobs=np.array([n_b, n_a]),
    value=0,
    alternative="two-sided"
)
print(f"Estadístico Z: {z_stat:.4f}")
print(f"P-value:      {p_value:.6g}")
print("Decisión:", "Rechazar H0" if p_value < ALPHA else "No rechazar H0")
print("Dirección observada:", "B > A" if difference > 0 else "B < A" if difference < 0 else "B = A")

## 8. Cálculo manual y comprobación

Hacemos el cálculo sin funciones especiales para explicar cada parte y verificar consistencia.

In [ ]:
pooled = (x_a+x_b)/(n_a+n_b)
se_null = np.sqrt(pooled*(1-pooled)*(1/n_a+1/n_b))
z_manual = difference/se_null
p_manual = 2*norm.sf(abs(z_manual))
print(f"Proporción agrupada: {pooled:.6f}")
print(f"Error estándar H0:   {se_null:.6f}")
print(f"Z manual: {z_manual:.5f} | p manual: {p_manual:.8g}")
assert np.allclose([z_manual,p_manual], [z_stat,p_value], rtol=1e-9)
print("✓ Verificación superada")

**Ejercicio 2:** Si el p-value es mayor que 0.05, ¿es correcto afirmar que ambas páginas convierten exactamente igual? Explica por qué no.

## 9. Intervalo de confianza del 95% para la diferencia B − A

Construimos el intervalo de **Newcombe**, apropiado para una diferencia de proporciones independientes. Para contraste bilateral a 5%, revisa si incluye cero. No utilices el solapamiento de dos intervalos individuales como sustituto del intervalo de diferencia.

In [ ]:
ci_low, ci_high = confint_proportions_2indep(
    count1=x_b, nobs1=n_b,
    count2=x_a, nobs2=n_a,
    compare="diff", method="newcomb", alpha=ALPHA
)
print(f"Diferencia = {difference*100:+.4f} pp")
print(f"IC 95% de B-A = [{ci_low*100:+.4f}, {ci_high*100:+.4f}] pp")
print("Contiene cero:", ci_low <= 0 <= ci_high)
print("Nota: p-value (Z agrupada) e IC (Newcombe) pueden discrepar muy cerca del umbral porque usan métodos distintos.")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.7))
ax.errorbar([100*difference], [0],
            xerr=np.array([[100*(difference-ci_low)], [100*(ci_high-difference)]]),
            fmt="o", capsize=6, markersize=8)
ax.axvline(0, color="black", linestyle="--", linewidth=1, label="Efecto nulo")
ax.axvline(100*MDE_ABSOLUTE, color="gray", linestyle=":", label="MDE ilustrativa (+1 pp)")
ax.set(yticks=[], xlabel="Diferencia B − A (puntos porcentuales)",
       title="Efecto observado e intervalo de confianza (Newcombe, 95%)")
ax.legend(); fig.tight_layout()
plot_path = OUTPUT_DIR / "diferencia_conversion_ic95.png"
fig.savefig(plot_path, dpi=160, bbox_inches="tight")
plt.show()

## 10. Significancia estadística y relevancia práctica

- **Significancia estadística:** evidencia contra igualdad de conversiones según un contraste predefinido.
- **Relevancia práctica:** ¿el efecto posible es suficiente para justificar el cambio?
- Para demostrar una mejora de **al menos** 1 pp hacen falta criterios específicos y un diseño que permita evaluar ese umbral. Un test contra cero no basta.

La MDE fijada en este notebook es pedagógica, no una decisión real de negocio.

In [ ]:
print(f"MDE de ejemplo: {100*MDE_ABSOLUTE:.2f} pp")
if ci_low > MDE_ABSOLUTE:
    print("Todo el IC está por encima de la mejora mínima ilustrativa.")
elif ci_high < MDE_ABSOLUTE:
    print("Todo el IC está por debajo de la mejora mínima ilustrativa.")
else:
    print("El IC cruza la MDE: no podemos clasificar con certeza ese umbral de mejora.")
if ci_high < 0:
    print("El intervalo respalda un efecto negativo.")
elif ci_low > 0:
    print("El intervalo respalda un efecto positivo.")
else:
    print("El intervalo es compatible con deterioro y/o mejora, e incluye efecto nulo.")

## 11. Sensibilidad: potencia para un efecto de interés

Esta potencia calculada usando la tasa de control observada y la MDE hipotética es **orientativa**. Un diseño confirmatorio requiere potencia y tamaño de muestra calculados *antes* de recolectar datos. Evita interpretar potencia post-hoc basada en el efecto observado como evidencia adicional.

In [ ]:
analysis = NormalIndPower()
p_alt = min(p_a + MDE_ABSOLUTE, 0.999999)
if p_alt > p_a:
    es = abs(proportion_effectsize(p_alt, p_a))
    power_reference = analysis.power(effect_size=es, nobs1=n_a, alpha=ALPHA,
                                     ratio=n_b/n_a, alternative="two-sided")
    n_required = analysis.solve_power(effect_size=es, alpha=ALPHA, power=0.80,
                                       ratio=1, alternative="two-sided")
    print(f"Potencia aproximada con muestra actual para +{MDE_ABSOLUTE*100:.2f} pp: {power_reference:.1%}")
    print(f"Tamaño orientativo por grupo (80% potencia, ratio 1:1): {np.ceil(n_required):,.0f}")
else:
    print("Revisa la tasa base y MDE")

## 12. Contraste de robustez: chi-cuadrado 2×2

La prueba de independencia chi-cuadrado de Pearson, sin corrección de Yates, es equivalente al cuadrado de la prueba Z de dos proporciones bajo los mismos supuestos. **No se considera una segunda prueba independiente**; se usa para comprobar la implementación.

In [ ]:
table_2x2 = np.array([[x_b,n_b-x_b],[x_a,n_a-x_a]])
chi2, p_chi2, dof, expected = chi2_contingency(table_2x2, correction=False)
print(f"Chi² = {chi2:.6f} | Z² = {z_stat*z_stat:.6f}")
print(f"p chi² = {p_chi2:.8g} | p Z = {p_value:.8g}")
assert np.isclose(chi2,z_stat*z_stat,rtol=1e-8)
assert np.isclose(p_chi2,p_value,rtol=1e-8)

## 13. Conclusión programática (no reemplaza criterio profesional)

Las afirmaciones que siguen describen la **muestra limpia**. Para un lanzamiento real revisar: diseño, SRM en datos brutos, exclusiones, métrica predefinida, experimentos simultáneos, posibles múltiples comparaciones y costos.

In [ ]:
if p_value < ALPHA:
    decision = "Se rechaza H0 al 5% (diferencia estadísticamente detectable)."
else:
    decision = "No se rechaza H0 al 5% (evidencia insuficiente para detectar una diferencia)."
if ci_low > 0:
    direction = "El IC 95% apunta a una mejora de B frente a A."
elif ci_high < 0:
    direction = "El IC 95% apunta a una reducción con B."
else:
    direction = "El IC 95% incluye cero; no se ha descartado la ausencia de efecto."
conclusion = (f"Conversión A={p_a:.4%}, B={p_b:.4%}; diferencia B-A="
              f"{difference*100:+.4f} pp; lift={lift*100:+.2f}%; "
              f"Z={z_stat:.3f}; p={p_value:.5g}; "
              f"IC 95% Newcombe=[{ci_low*100:+.4f}, {ci_high*100:+.4f}] pp. "
              f"{decision} {direction} "
              "Conclusión exploratoria condicionada a la limpieza e hipótesis de independencia.")
print(conclusion)

## 14. Guardar resultados en CSV / JSON / TXT y opcionalmente S3

Se exportan artefactos para el Notebook 04 y para el informe final. Confirma permisos IAM `s3:PutObject`. Se mantienen las tasas como proporciones en el CSV.

In [ ]:
results = pd.DataFrame([{
    "n_control": n_a, "n_treatment": n_b,
    "converted_control": x_a, "converted_treatment": x_b,
    "rate_control": p_a, "rate_treatment": p_b,
    "difference_b_minus_a": difference, "relative_lift": lift,
    "z_statistic": z_stat, "p_value_two_sided": p_value,
    "alpha": ALPHA, "ci95_newcombe_low": ci_low,
    "ci95_newcombe_high": ci_high, "srm_p_value": srm_p,
    "mde_absolute_example": MDE_ABSOLUTE
}])
display(results.T)
results.to_csv(OUTPUT_DIR / "hypothesis_test_results.csv", index=False)
(OUTPUT_DIR / "conclusion.txt").write_text(conclusion + "\n", encoding="utf-8")
(OUTPUT_DIR / "metadata.json").write_text(json.dumps({
    "notebook": "03", "dataset": INPUT_KEY if USE_S3 else str(LOCAL_PATH),
    "test": "two-sided two-proportion pooled z-test",
    "confidence_interval": "Newcombe difference, 95%",
    "warning": "Cleaned sample is not equivalent to intention-to-treat"
}, ensure_ascii=False, indent=2), encoding="utf-8")
if USE_S3:
    for filepath in OUTPUT_DIR.iterdir():
        if filepath.is_file():
            dest = RESULTS_KEY_PREFIX + filepath.name
            s3.upload_file(str(filepath), BUCKET, dest)
            print(f"s3://{BUCKET}/{dest}")
else:
    print("Resultados locales:", OUTPUT_DIR.resolve())

## 15. Ejercicios de interpretación

1. Reporta tasas, diferencia en **puntos porcentuales** y lift relativo. ¿Podría un pequeño lift ser estadísticamente significativo sin ser valioso económicamente?
2. Explica la diferencia entre $P(\text{datos extremos}\mid H_0)$ y $P(H_0\mid\text{datos})$.
3. ¿Por qué no podemos afirmar que los grupos son equivalentes cuando p > 0.05?
4. Repite el gráfico cambiando `MDE_ABSOLUTE = 0.005`. **No** cambies el umbral para lograr una recomendación deseada.
5. Imagina que el dataset original perdió 5% de observaciones solo en tratamiento. ¿Cómo afectaría la validez causal?
6. ¿Qué otras métricas guardrail revisarías (ingresos por usuario, latencia, reclamaciones)?

### Checklist de salida

- [ ] `user_id` único y variables validadas
- [ ] Balance y SRM revisados respecto de asignación planeada
- [ ] Diferencia absoluta y lift explicados
- [ ] Prueba Z bilateral y comprobación manual
- [ ] IC de diferencia + gráfica
- [ ] Relevancia práctica separada de significancia
- [ ] Resultados exportados

**Siguiente:** Notebook 04 — Bootstrap, simulación Monte Carlo y potencia estadística.

> **Nota:** si este experimento fuera destinado a producción, añade una auditoría de los datos *antes* de la limpieza y análisis ITT basado en asignación original. Evita decidir a partir de inspecciones reiteradas del p-value durante el experimento sin un diseño secuencial apropiado.